## gCRL-AE applied on the Norman2019 dataset

- Load processed Norman2019 dataset
- Compute eigengenes
- Train gCRL-AE
- Partial MCC permutation analysis

In [1]:
# ensuring all packages are reloaded each time I run a cell
%load_ext autoreload
%autoreload 2

In [2]:
import os
import itertools

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr, spearmanr, ttest_1samp

from gcrl.training.train_gcrl_ae import train_gcrl_ae
from gcrl.grn.eigengenes import compute_eigengenes
from gcrl.alignment.partial_mcc_perm_experiments import run_partial_mcc_perm_experiments


In [3]:
# ── Control panel ────────────────────────────────────────────────

# --- dataset parameters ---
gamma = '1p1'   # '0p9' | '1p0' | '1p4' | '2p5'
pct   = 50      # percentage threshold used during preprocessing

# --- paths (derived from dataset parameters) ---
input_data_folder = '../../data/real/Norman2019/'
input_data_file   = f'{input_data_folder}Norman2019_processed_{pct}pct_gamma{gamma}.h5ad'
output_ae_dir     = f'../../results/real/Norman2019/ae_{pct}pct_gamma{gamma}'
output_mcc_dir    = f'../../results/real/Norman2019/partial_mcc_permutation_{pct}pct_gamma{gamma}'

# --- eigengene computation ---
# mode: 'global'    — PCA fitted on reference cells, projected to all cells
#        'all_cells' — PCA fitted and projected on all cells (no reference subset)
eigengene_mode  = 'all_cells'  # 'global' | 'all_cells'
reference_query = 'intervention == "unperturbed"'

# --- AE architecture ---
ae_input_mode      = 'TF'     # 'TF' | 'ALL'
ae_reconstruct_all = True     # True: decoder reconstructs all genes
ae_latent_dim      = None     # None → inferred as (#communities + 1)
ae_hidden_dims     = (64,)    # encoder MLP hidden layer sizes

# --- AE training ---
ae_standardize   = 'zscore'  # 'zscore_ref' | 'minmax_0_1' | 'none' | 'zscore'
ae_batch_size    = 1024
ae_lr            = 1e-3
ae_num_epochs    = 100
ae_lr_step       = 100
ae_weight_decay  = 1e-3
ae_val_frac      = 0.1

# --- partial-MCC permutation experiment ---
mcc_n_real_seeds   = 100
mcc_n_permutations = 100
mcc_n_perm_seeds   = 3
mcc_lr             = 5e-2
mcc_steps          = 100
mcc_master_seed    = 123
mcc_include_pooled = True


In [4]:
adata = sc.read_h5ad(input_data_file)
adata


AnnData object with n_obs × n_vars = 35048 × 2070
    obs: 'guide_identity', 'read_count', 'UMI_count', 'gemgroup', 'good_coverage', 'number_of_cells', 'guide_ids', 'guide_merged', 'split', 'batch', 'condition', 'cell_type', 'dose_val', 'control', 'drug_dose_name', 'cov_drug_dose_name', 'intervention', 'set'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensembl', 'kind', 'community'
    uns: 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [5]:
compute_eigengenes(
    adata,
    mode=eigengene_mode,
    reference_query=reference_query,
)
adata


AnnData object with n_obs × n_vars = 35048 × 2070
    obs: 'guide_identity', 'read_count', 'UMI_count', 'gemgroup', 'good_coverage', 'number_of_cells', 'guide_ids', 'guide_merged', 'split', 'batch', 'condition', 'cell_type', 'dose_val', 'control', 'drug_dose_name', 'cov_drug_dose_name', 'intervention', 'set'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensembl', 'kind', 'community'
    uns: 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov', 'umap', 'X_comm_eig_comm_ids', 'X_comm_eig_global_index', 'comm_eig_meta'
    obsm: 'X_pca', 'X_umap', 'X_comm_eig'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [6]:
meta     = adata.uns['comm_eig_meta']
comm_ids = adata.uns['X_comm_eig_comm_ids']

# Build a flat per-community stats dict regardless of eigengene mode.
# 'global' and 'all_cells' store stats at the top level;
# 'by_cell_type' stores them nested under per_cell_type — average across types.
if meta['mode'] in ('global', 'all_cells'):
    flat_stats  = meta['per_comm_stats']
    flat_genes  = meta['per_comm_genes']
    flat_pooled = meta['pooled_tf_stats']['explained_variance_ratio']
else:  # by_cell_type
    ct_metas = list(meta['per_cell_type'].values())
    flat_stats = {
        str(c): {'explained_variance_ratio': np.mean(
            [ct['per_comm_stats'][str(c)]['explained_variance_ratio'] for ct in ct_metas]
        )}
        for c in comm_ids
    }
    flat_genes  = ct_metas[0]['per_comm_genes']
    flat_pooled = np.mean([ct['pooled_tf_stats']['explained_variance_ratio'] for ct in ct_metas])

rows = []
for comm in comm_ids:
    rows.append({
        'community': comm,
        'n_tfs': len(flat_genes[str(comm)]),
        'explained_variance_ratio': flat_stats[str(comm)]['explained_variance_ratio'],
    })
rows.append({'community': 'all_TF',
             'n_tfs': sum(len(flat_genes[str(c)]) for c in comm_ids),
             'explained_variance_ratio': flat_pooled})

df_evr   = pd.DataFrame(rows).set_index('community')
comm_only = df_evr.drop(index='all_TF', errors='ignore')
r_p, p_p = pearsonr(comm_only['n_tfs'], comm_only['explained_variance_ratio'])
r_s, p_s = spearmanr(comm_only['n_tfs'], comm_only['explained_variance_ratio'])

df_evr['explained_variance_ratio'] = (
    df_evr['explained_variance_ratio'] * 100
).round(1).astype(str) + '%'
print(df_evr.to_string())
print(f'\nCorrelation n_tfs vs EVR (communities only):')
print(f'  Pearson  r = {r_p:+.3f}  (p = {p_p:.3g})')
print(f'  Spearman r = {r_s:+.3f}  (p = {p_s:.3g})')


           n_tfs explained_variance_ratio
community                                
0.0           28                     5.1%
1.0           27                     5.1%
2.0           20                     7.1%
3.0           23                     5.0%
4.0           18                     9.7%
5.0           12                    10.2%
6.0            2                    50.0%
8.0           11                    12.6%
all_TF       141                     2.6%

Correlation n_tfs vs EVR (communities only):
  Pearson  r = -0.822  (p = 0.0123)
  Spearman r = -0.929  (p = 0.000863)


In [7]:
interventions = adata.obs['intervention'].unique()
perturbed_tfs = set(
    itertools.chain.from_iterable(i.split('+') for i in interventions if i != 'unperturbed')
)

tf_var = adata.var.loc[adata.var['kind'] == 'TF', ['community']].dropna()
perturbed_tf_var = tf_var[tf_var.index.isin(perturbed_tfs)]

community_to_tfs = (
    perturbed_tf_var.reset_index()
    .groupby('community', observed=True)['gene_symbols']
    .apply(sorted)
    .apply(list)
)

print(f'Unique perturbed TFs: {len(perturbed_tfs)}')
print(f'Communities targeted: {sorted(community_to_tfs.index.astype(int).tolist())}\n')
for comm, tfs in community_to_tfs.items():
    print(f'  Community {int(comm):2d}  ({len(tfs):2d} perturbed TFs): {", ".join(tfs)}')


Unique perturbed TFs: 30
Communities targeted: [0, 1, 2, 3, 4, 5, 8]

  Community  0  ( 8 perturbed TFs): CEBPE, ETS2, FEV, FOXA1, FOXA3, FOXF1, HES7, HOXA13
  Community  1  ( 5 perturbed TFs): AHR, CEBPA, FOSB, HOXC13, KLF1
  Community  2  ( 4 perturbed TFs): EGR1, HNF4A, LHX1, MEIS1
  Community  3  ( 4 perturbed TFs): HOXB9, IRF1, POU3F2, TBX2
  Community  4  ( 4 perturbed TFs): CEBPB, DLX2, FOXL2, JUN
  Community  5  ( 3 perturbed TFs): LYL1, OSR2, PRDM1
  Community  8  ( 2 perturbed TFs): SPI1, TBX3


## gCRL-AE training

In [8]:
res = train_gcrl_ae(
    adata=adata,
    input_mode=ae_input_mode,
    reconstruct_all=ae_reconstruct_all,
    latent_dim=ae_latent_dim,
    hidden_dims=ae_hidden_dims,
    standardize=ae_standardize,
    reference_query=reference_query,
    batch_size=ae_batch_size,
    lr=ae_lr,
    num_epochs=ae_num_epochs,
    lr_step=ae_lr_step,
    weight_decay=ae_weight_decay,
    val_frac=ae_val_frac,
    device=None,
    outdir=output_ae_dir,
)


In [9]:
loss     = res.history['loss']
val_loss = res.history.get('val_loss')

header = f"{'Epoch':>6}  {'Train loss':>12}  {'Val loss':>12}"
print(header)
print('-' * len(header))
for ep in [1, 10, 50, 100, 200]:
    if ep > len(loss):
        break
    vl = f'{val_loss[ep-1]:12.4f}' if val_loss else '           -'
    print(f'{ep:6d}  {loss[ep-1]:12.4f}  {vl}')

print()
print(f'Train improvement: {loss[0]-loss[-1]:.4f}  ({100*(loss[0]-loss[-1])/loss[0]:.1f}% reduction)')
if val_loss:
    print(f'Val   improvement: {val_loss[0]-val_loss[-1]:.4f}  ({100*(val_loss[0]-val_loss[-1])/val_loss[0]:.1f}% reduction)')
    gap = val_loss[-1] - loss[-1]
    print(f'Final val - train gap: {gap:+.4f}  ({"possible overfitting" if gap > 0.25 else "no significant overfitting"})')


 Epoch    Train loss      Val loss
----------------------------------
     1        1.0109        1.0383
    10        0.9673        1.0047
    50        0.9467        0.9905
   100        0.9434        0.9882

Train improvement: 0.0675  (6.7% reduction)
Val   improvement: 0.0501  (4.8% reduction)
Final val - train gap: +0.0448  (no significant overfitting)


In [10]:
B = res.embeddings
print(B.shape)


(35048, 9)


In [11]:
os.makedirs(output_mcc_dir, exist_ok=True)

out = run_partial_mcc_perm_experiments(
    adata=adata,
    embeddings=B,
    community_col='community',
    reference_query=reference_query,
    mode=eigengene_mode,
    n_real_seeds=mcc_n_real_seeds,
    n_permutations=mcc_n_permutations,
    n_perm_seeds=mcc_n_perm_seeds,
    include_pooled_tf=mcc_include_pooled,
    lr=mcc_lr,
    steps=mcc_steps,
    device=None,
    master_seed=mcc_master_seed,
    save_density_path=os.path.join(output_mcc_dir, 'dens.png'),
)

np.save(os.path.join(output_mcc_dir, 'scores_real.npy'), out['scores_real'])
np.save(os.path.join(output_mcc_dir, 'scores_perm.npy'), out['scores_perm'])
np.savez(os.path.join(output_mcc_dir, 'partial_mcc_results.npz'),
         scores_real=out['scores_real'],
         scores_perm=out['scores_perm'])
print(f'Results saved to {output_mcc_dir}')


Real runs:   0%|          | 0/100 [00:00<?, ?run/s]

Permutations:   0%|          | 0/100 [00:00<?, ?perm/s]

Results saved to ../../results/real/Norman2019/partial_mcc_permutation_50pct_gamma1p1


In [12]:
scores_real = out['scores_real']          # shape (n_real_seeds,)
scores_perm = out['scores_perm']          # shape (n_permutations, n_perm_seeds)

# reference: mean over real seeds
mu_real = scores_real.mean()

# distribution under null: per-permutation mean across perm seeds
perm_means = scores_perm.mean(axis=1)     # shape (n_permutations,)
avg_std_across_seeds = scores_perm.std(axis=1).mean()

print(f'Real   — mean: {mu_real:.4f}  std: {scores_real.std():.4f}  n={len(scores_real)}')
print(f'Perm means — mean: {perm_means.mean():.4f}  std: {perm_means.std():.4f}  n={len(perm_means)}')
print(f'Perm — avg std across seeds: {avg_std_across_seeds:.4f}')
print()

t_stat, p_val = ttest_1samp(perm_means, popmean=mu_real, alternative='less')
print(f'One-sample t-test (perm means < real mean): t = {t_stat:.4f},  p = {p_val:.4g}')

Real   — mean: 0.4743  std: 0.0031  n=100
Perm means — mean: 0.4733  std: 0.0409  n=100
Perm — avg std across seeds: 0.0017

One-sample t-test (perm means < real mean): t = -0.2486,  p = 0.4021
